# LFM Tiling Example

This notebook demonstrates the configuration-driven lunar tiling API. It creates and visualizes two mixed-modality examples on the LTM grid:

1. A product-scoped WAC cube plus the canonical 63-band static cube.
2. A product-scoped NAC cube plus the same static context.

All raster sources use existing, explicitly configured vector indexes and bilinear resampling. Results are returned as `TileCubeRecord` objects, so downstream code uses structured source, zone, zoom, and tile fields instead of parsing output filenames. The legacy notebook remains at `notebooks/toy_model/tiling_example.ipynb` during migration.

## Imports and repository discovery

Run this notebook from the repository's top-level `notebooks/` directory. JupyterHub may expose the clone through `/panfs`; repository discovery normalizes that path to the equivalent `/explore` symlink before importing LFM.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings

from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

In [ ]:
repo_root = Path.cwd().parent
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists() or not (repo_root / "model").exists():
    raise FileNotFoundError(
        "Cannot find the lfm/ and model/ directories. Run this notebook "
        "from the repository's top-level notebooks/ directory."
    )

sys.path.insert(0, str(repo_root))

from model import (
    TileConfig,
    TileSourceConfig,
    create_tiles_for_aoi,
    create_tiles_for_index,
    create_tiles_for_point,
)
from lfm.all_models.all_tasks.tiling_utils import (
    DEFAULT_NAC_BAND_NUMBER,
    DEFAULT_WAC_BAND_NUMBER,
    DEFAULT_ZOOM_LEVEL,
    RUN_ID,
    make_static_source,
    validate_path_pairs,
)
from lfm.all_models.all_tasks.viz import (
    pair_dynamic_and_static,
    plot_cube_pairs,
    print_record_summary,
)

print(f"Repository root: {repo_root}")
print("Successfully imported the modern LFM tiling API")

## User configuration

The defaults below use the representative Explore data exercised by the tiling validation suite. Change these paths and selectors for another dataset. Each modality declares its raster directory and existing `.shp` or `.gpkg` index explicitly; tiling never creates or refreshes an index.

Generated cubes and plots are written beneath `repo_root/outputs/tiling/<RUN_ID>/`. The timestamped run directory prevents one execution from silently reusing another execution's cubes. AOI queries discover the intersecting LTM zone or zones automatically. NAC coverage is sparse, so the NAC example may have fewer dynamic/static pairs than the WAC example.

In [ ]:
PROJECT_DATA_DIR = Path("/explore/nobackup/projects/lfm")
WAC_DATA_DIR = PROJECT_DATA_DIR / "processed_data/Lunar/LRO_WAC_Pho_Sites"
NAC_DATA_DIR = PROJECT_DATA_DIR / "processed_data/Lunar/LRO_NAC_Pho_Sites"
STATIC_DATA_DIR = PROJECT_DATA_DIR / "staticLinks"

WAC_PRODUCT_ID = "M1107459759CE"
NAC_PRODUCT_ID = "M1117899885LE"
WAC_AOI_BOUNDS = {
    "ul_lat": 31.241501479374616,
    "ul_lon": 121.01616204071722,
    "lr_lat": 30.201163639632075,
    "lr_lon": 122.22643208573085,
}
NAC_AOI_BOUNDS = {
    "ul_lat": 1.0786543156953,
    "ul_lon": 149.752054273755,
    "lr_lat": 1.0586543156953,
    "lr_lon": 149.772054273755,
}

## Setting up variable values

The values in this section are derived defaults and notebook controls. WAC uses the repository's standard zoom level, while NAC uses a near-native zoom appropriate for its 1 m source imagery. Most users can leave these values unchanged.

In [ ]:
WAC_INDEX = WAC_DATA_DIR / "output_index.shp"
NAC_INDEX = NAC_DATA_DIR / "output_index.shp"
STATIC_INDEX = STATIC_DATA_DIR / "db2.shp"
LOCATION_FIELD = "location"
WAC_ZOOM_LEVEL = DEFAULT_ZOOM_LEVEL
NAC_ZOOM_LEVEL = 11  # 1.185 m/pixel; close to the processed NAC resolution

BASE_OUTPUT_DIR = repo_root / "outputs" / "tiling"
OUTPUT_DIR = BASE_OUTPUT_DIR / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

# Plot WAC VIS band 0, the single NAC image band, and static elevation.
WAC_BAND_NUMBER = DEFAULT_WAC_BAND_NUMBER  # 1-based: first VIS channel
NAC_BAND_NUMBER = DEFAULT_NAC_BAND_NUMBER
STATIC_BAND_TO_PLOT = "lola_kaguya_60mpp_elv"
MAX_PLOT_TILES = 4
RUN_ALTERNATE_QUERIES = False

print(f"Notebook outputs will default here: {OUTPUT_DIR}")

## Resolve data paths

Validate the configured data directories and their indexes, then construct the shared static source configuration.

In [ ]:
validate_path_pairs(
    {
        "WAC data directory": WAC_DATA_DIR,
        "NAC data directory": NAC_DATA_DIR,
        "static data directory": STATIC_DATA_DIR,
    },
    path_type="directory",
)
validate_path_pairs(
    {
        "WAC index": WAC_INDEX,
        "NAC index": NAC_INDEX,
        "static index": STATIC_INDEX,
    },
    path_type="file",
)

STATIC_SOURCE = make_static_source(
    data_dir=STATIC_DATA_DIR,
    index_path=STATIC_INDEX,
    location_field=LOCATION_FIELD,
)

## Example 1: WAC + STATIC AOI tiling

WAC uses `product_id` selection and preserves its native per-band NoData metadata. Static uses `all_intersecting`, the canonical band order, and standardized output NoData. Source order controls the order of records returned within each tile.

In [ ]:
wac_source = TileSourceConfig(
    name="wac",
    data_dir=WAC_DATA_DIR,
    index_path=WAC_INDEX,
    location_field=LOCATION_FIELD,
    selection_mode="product_id",
    resampling="bilinear",
    preserve_source_nodata=True,
)
wac_static_config = TileConfig(
    output_dir=OUTPUT_DIR / "wac_static",
    zoom_level=WAC_ZOOM_LEVEL,
    sources=(wac_source, STATIC_SOURCE),
)
wac_static_config

In [ ]:
wac_static_records = create_tiles_for_aoi(
    wac_static_config,
    **WAC_AOI_BOUNDS,
    selectors={"wac": WAC_PRODUCT_ID},
)
print_record_summary(wac_static_records)

wac_static_pairs = pair_dynamic_and_static(wac_static_records, "wac")
print(f"WAC/static pairs available for plotting: {len(wac_static_pairs)}")

In [ ]:
wac_figure = plot_cube_pairs(
    wac_static_pairs,
    dynamic_label="WAC",
    dynamic_band_number=WAC_BAND_NUMBER,
    static_band_name=STATIC_BAND_TO_PLOT,
    output_path=OUTPUT_DIR / "plots" / "wac_static_cubes.png",
    max_tiles=MAX_PLOT_TILES,
)

## Example 2: NAC + STATIC AOI tiling

The same public API handles NAC without a WAC alias. This example uses LTM zoom 11 (approximately 1.185 m/pixel) to stay close to the processed NAC source's 1 m resolution and avoid placing its narrow footprint inside a much larger zoom-5 tile. NAC is declared `required=False` because the selected observation is sparse: an AOI tile without that NAC product is an expected skip, while static context is still produced for every intersecting tile. Plot pairing uses only tiles containing both modalities.

In [ ]:
nac_source = TileSourceConfig(
    name="nac",
    data_dir=NAC_DATA_DIR,
    index_path=NAC_INDEX,
    location_field=LOCATION_FIELD,
    selection_mode="product_id",
    resampling="bilinear",
    preserve_source_nodata=True,
    required=False,
)
nac_static_config = TileConfig(
    output_dir=OUTPUT_DIR / "nac_static",
    zoom_level=NAC_ZOOM_LEVEL,
    sources=(nac_source, STATIC_SOURCE),
)
nac_static_config

In [ ]:
nac_static_records = create_tiles_for_aoi(
    nac_static_config,
    **NAC_AOI_BOUNDS,
    selectors={"nac": NAC_PRODUCT_ID},
)
print_record_summary(nac_static_records)

nac_static_pairs = pair_dynamic_and_static(nac_static_records, "nac")
print(f"NAC/static pairs available for plotting: {len(nac_static_pairs)}")

In [ ]:
nac_figure = plot_cube_pairs(
    nac_static_pairs,
    dynamic_label="NAC",
    dynamic_band_number=NAC_BAND_NUMBER,
    static_band_name=STATIC_BAND_TO_PLOT,
    output_path=OUTPUT_DIR / "plots" / "nac_static_cubes.png",
    max_tiles=MAX_PLOT_TILES,
)

## Alternative query entry points

The examples above use a geographic AOI. The same `TileConfig` contract supports a point query and an explicit LTM tile-index query. Set `RUN_ALTERNATE_QUERIES = True` in the user configuration to run the examples below. Separate output directories prevent these calls from overwriting the AOI outputs.

In [ ]:
if RUN_ALTERNATE_QUERIES:
    example_wac_record = next(
        record for record in wac_static_records if record.source_name == "wac"
    )
    example_lat = (WAC_AOI_BOUNDS["ul_lat"] + WAC_AOI_BOUNDS["lr_lat"]) / 2
    example_lon = (WAC_AOI_BOUNDS["ul_lon"] + WAC_AOI_BOUNDS["lr_lon"]) / 2

    point_config = TileConfig(
        output_dir=OUTPUT_DIR / "wac_static_point",
        zoom_level=WAC_ZOOM_LEVEL,
        sources=wac_static_config.sources,
    )
    point_records = create_tiles_for_point(
        point_config,
        lat=example_lat,
        lon=example_lon,
        zone=example_wac_record.zone,
        selectors={"wac": WAC_PRODUCT_ID},
    )
    print("Point-query records:")
    print_record_summary(point_records)

    index_config = TileConfig(
        output_dir=OUTPUT_DIR / "wac_static_tile_index",
        zoom_level=WAC_ZOOM_LEVEL,
        sources=wac_static_config.sources,
    )
    index_records = create_tiles_for_index(
        index_config,
        tile_x=example_wac_record.tile_x,
        tile_y=example_wac_record.tile_y,
        zone=example_wac_record.zone,
        selectors={"wac": WAC_PRODUCT_ID},
    )
    print("Tile-index-query records:")
    print_record_summary(index_records)
else:
    print("Alternate point and tile-index queries are configured but disabled.")

## Outputs

The run directory contains separate WAC/static and NAC/static cube directories plus saved PNG visualizations. Each returned record contains the authoritative modality and tile identity; filenames remain descriptive for human inspection but are not the machine-readable interface.

In [ ]:
print(f"Completed tiling notebook run: {OUTPUT_DIR}")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUTPUT_DIR)}")